# Demo — RAG em grafo: ligar fatos de documentos diferentes

**Objetivo:** mostrar o **degrau 4** do slide 17 ("Falha em pergunta que exige ligar fatos de documentos diferentes" → Grafo) e a linha **Grafo** do slide 15: percorrer entidades e relações do acervo, com indexação cara e GraphRAG fora do AI Search. Vale a regra do slide 13: só se sobe para cá por sintoma medido.

1. **O limite do híbrido com reranking** → a melhor busca do índice traz pedaços da resposta, mas não a cadeia inteira;
2. **Indexação do grafo (a parte cara)** → uma chamada ao modelo por documento extrai entidades e relações;
3. **Consulta no grafo** → da entidade citada na pergunta até os projetos, salto a salto;
4. **Resposta com o grafo** → o caminho e os trechos de onde ele saiu viram o contexto, com citação.

> **Cenário 100% fictício.** A Contoso Industrial e todos os documentos, projetos, pessoas e números do `acervo/` existem apenas para fins didáticos.

**Pré-requisito:** o índice `cap05-acervo`, criado e carregado pelo notebook `tipos-de-rag/00-indice-compartilhado` (usado na Execução 1). O grafo é montado aqui mesmo, em memória, com `networkx`: este notebook não cria nenhum recurso no Azure.

## 0. Preparação

Variáveis necessárias:

- `AZURE_OPENAI_ENDPOINT` e `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_DEPLOYMENT` (deployment de chat, usado na extração do grafo e nas respostas): precisa aceitar **structured outputs**, como o `gpt-5-mini`
- `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`, `AZURE_OPENAI_EMBEDDING_MODEL` e `AZURE_OPENAI_EMBEDDING_DIMENSIONS`: os **mesmos** valores usados para criar o índice no notebook 00
- `AZURE_SEARCH_ENDPOINT` e `AZURE_SEARCH_API_KEY`

Os valores vêm do arquivo `Cap05/.env` (copie o `.env.example` e preencha); o que faltar é perguntado na hora da execução.

A Execução 1 faz uma consulta com o semantic ranker, no mesmo AI Search do notebook 00 (tier Basic ou superior). Se ela der erro (por exemplo, com a cota mensal do plano gratuito esgotada), siga a partir da Execução 2 e, antes da Execução 4, rode uma célula com `ids_top5 = []`: a comparação da Execução 4 vai mostrar 0/4 para a Execução 1.

A Execução 2 salva o grafo em `grafo.json`, nesta pasta. Nas execuções seguintes ele é lido do arquivo, sem chamar o modelo de novo: apague `grafo.json` para reextrair.

> Não grave chaves diretamente no notebook.

In [ ]:
%pip install -q "openai>=2" "python-dotenv>=1.0" "pandas>=2.1" "pydantic>=2.7" "networkx>=3.2" "matplotlib>=3.8" "azure-search-documents>=12.0.0,<13"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))  # lê o .env da pasta Cap05, se existir

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or input(
    "AZURE_OPENAI_ENDPOINT (ex.: https://meu-recurso.openai.azure.com): "
).strip()
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY") or getpass("AZURE_OPENAI_API_KEY: ")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or input(
    "AZURE_OPENAI_DEPLOYMENT (deployment de chat): "
).strip()
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT") or input(
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT (deployment de embedding): "
).strip()

# O modelo e a dimensão do embedding são o contrato entre indexação e consulta.
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", "1536"))

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# Pasta Cap05: a primeira pasta, subindo a partir daqui, que contém o acervo.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "acervo").is_dir())
PASTA_ACERVO = RAIZ / "acervo"

print("Cliente configurado.")
print("Deployment de chat:", AZURE_OPENAI_DEPLOYMENT)
print("Embedding:", AZURE_OPENAI_EMBEDDING_DEPLOYMENT, f"({EMBEDDING_MODEL}, {EMBEDDING_DIMENSIONS} dimensões)")
print("Acervo:", PASTA_ACERVO)

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT") or input(
    "AZURE_SEARCH_ENDPOINT (ex.: https://meu-servico.search.windows.net): "
).strip()
AZURE_SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY") or getpass("AZURE_SEARCH_API_KEY (chave de administração): ")

NOME_INDICE = "cap05-acervo"
credencial_busca = AzureKeyCredential(AZURE_SEARCH_API_KEY)
index_client = SearchIndexClient(AZURE_SEARCH_ENDPOINT, credencial_busca)
search_client = SearchClient(AZURE_SEARCH_ENDPOINT, NOME_INDICE, credencial_busca)

# Este notebook só consulta o índice: quem cria e carrega é o notebook 00.
if NOME_INDICE not in list(index_client.list_index_names()):
    raise RuntimeError("Índice não encontrado. Rode antes o notebook tipos-de-rag/00-indice-compartilhado.")

print(f"Índice '{NOME_INDICE}' encontrado, com {search_client.get_document_count()} trechos.")

### Funções auxiliares

- `carregar_trechos`: o mesmo chunking do notebook 00, um trecho por seção. O grafo é extraído destes mesmos trechos, e cada relação guarda o `id` do trecho de onde saiu.
- `gerar_embeddings` e `responder`: os mesmos dos notebooks anteriores. O `responder` não muda em nenhuma execução: o que muda é o contexto que ele recebe.
- `PERGUNTAS` e `cobertura`: no gabarito, a pergunta `multissalto` tem quatro grupos de trechos esperados, um por documento da cadeia; `cobertura` conta quantos deles chegaram ao contexto.

In [ ]:
import re


def carregar_trechos(pasta_acervo: Path) -> list[dict]:
    """Lê os .md do acervo e devolve um trecho por seção "## " (chunking pela estrutura do documento)."""
    trechos = []
    for caminho in sorted(pasta_acervo.rglob("*.md")):
        fonte = caminho.relative_to(pasta_acervo).as_posix()
        doc_id = fonte.removesuffix(".md").replace("/", "_")
        texto = caminho.read_text(encoding="utf-8")
        titulo = texto.splitlines()[0].lstrip("# ").strip()
        # O que vem antes do primeiro "## " é o cabeçalho do documento (título e metadados).
        for numero, secao in enumerate(re.split(r"(?m)^## ", texto)[1:], start=1):
            nome_secao, _, conteudo = secao.partition("\n")
            trechos.append({
                "id": f"{doc_id}_{numero:02d}",
                "doc_id": doc_id,
                "titulo": titulo,
                "secao": nome_secao.strip(),
                "conteudo": conteudo.strip(),
                "fonte": fonte,
            })
    return trechos


trechos = carregar_trechos(PASTA_ACERVO)
print(f"{len(trechos)} trechos de {len({t['doc_id'] for t in trechos})} documentos")

In [ ]:
import json

pd.set_option("display.max_colwidth", None)  # tabelas mostram o texto inteiro


def gerar_embeddings(textos: list[str], lote: int = 16) -> list[list[float]]:
    vetores = []
    for i in range(0, len(textos), lote):
        resposta = client.embeddings.create(
            model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
            input=textos[i:i + lote],
            dimensions=EMBEDDING_DIMENSIONS,
        )
        vetores.extend(item.embedding for item in resposta.data)
    return vetores


INSTRUCOES_RAG = """
Responda em português, exclusivamente com base nos trechos fornecidos.
Cite a fonte de cada afirmação no formato [id do trecho].
Se os trechos não trouxerem a informação, responda:
'Não há evidência suficiente nos trechos recuperados.'
""".strip()


def responder(pergunta: str, trechos_recuperados: list[dict]) -> str:
    contexto = "\n\n".join(
        f"[{t['id']}] {t['titulo']} — {t['secao']}\n{t['conteudo']}" for t in trechos_recuperados
    )
    resposta = client.responses.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES_RAG,
        input=f"TRECHOS RECUPERADOS:\n{contexto}\n\nPERGUNTA:\n{pergunta}",
        store=False,
    )
    return resposta.output_text


PERGUNTAS = {p["id"]: p for p in json.loads((PASTA_ACERVO / "perguntas.json").read_text(encoding="utf-8"))}


def cobertura(ids_retornados: list[str], grupos: list[list[str]]) -> str:
    """Quantos dos assuntos esperados (grupos de trechos) apareceram na lista retornada."""
    achados = sum(any(i in ids_retornados for i in grupo) for grupo in grupos)
    return f"{achados}/{len(grupos)}"

---

# Execução 1 — O limite do híbrido com reranking

A pergunta é a `multissalto` do gabarito: *"A suspensão da Rolamentos Alfa coloca algum projeto da empresa em risco?"*. A resposta está no acervo, mas espalhada por quatro documentos, e nenhum trecho a contém sozinho:

| Elo | Documento (seção) | O que ele diz |
|---|---|---|
| 1 | Fornecedores homologados (Rolamentos Alfa Ltda.) | A Rolamentos Alfa é a fornecedora exclusiva do SKU 48213-B |
| 2 | Plano de manutenção preventiva (Prensa P-400) | A troca preventiva da Prensa P-400 usa o SKU 48213-B |
| 3 | Projeto ATLAS (Dependências e riscos) | O projeto depende da P-400 operando em três turnos |
| 4 | Comunicado de Compras (Suspensão da Rolamentos Alfa Ltda.) | A Rolamentos Alfa está suspensa, e novos pedidos ficam bloqueados |

Primeiro, a melhor busca que o índice oferece: **híbrida** (BM25 + vetor, fusão RRF) com o **semantic ranker** reordenando o resultado, como no notebook 03. A consulta vetorial pede `k = 50` vizinhos, como recomenda a documentação: o semantic ranker reordena até 50 resultados, e um `k` menor o deixa com menos candidatos. Os 5 primeiros vão para o `responder`.

Na tabela, o **score RRF** é o da fusão (números pequenos, por definição) e o **reranker** vai de 0 a 4: é ele que decide a ordem.

### O que esperar e por que pode variar

- Os trechos que citam a Rolamentos Alfa (o comunicado e a lista de fornecedores) tendem a aparecer no topo.
- As seções "Dependências e riscos" dos três projetos falam de projeto e de risco, como a pergunta, e disputam as outras vagas.
- O plano de manutenção da P-400 tende a ficar de fora: não tem nenhuma palavra da pergunta e trata de outro assunto (a troca preventiva do conjunto de apoio do volante a cada 2.000 horas de operação, com o item SKU 48213-B). E é justamente ele o elo entre o fornecedor e o projeto.
- O resultado depende do modelo de embedding e do ranker. Se o elo vier, leia a resposta mesmo assim: o modelo ainda precisa montar a cadeia sozinho, a partir de trechos soltos.

In [ ]:
from azure.search.documents.models import VectorizedQuery


def buscar_hibrido_com_reranking(pergunta: str, top: int = 5) -> list[dict]:
    """Híbrida (BM25 + vetor, fusão RRF) com o semantic ranker reordenando os 50 primeiros."""
    vetor = gerar_embeddings([pergunta])[0]
    resultados = search_client.search(
        search_text=pergunta,
        vector_queries=[VectorizedQuery(vector=vetor, k_nearest_neighbors=50, fields="embedding")],
        query_type="semantic",
        semantic_configuration_name="semantica-padrao",
        select=["id", "titulo", "secao", "conteudo", "fonte"],
        top=top,
    )
    return [
        {
            "id": r["id"],
            "titulo": r["titulo"],
            "secao": r["secao"],
            "conteudo": r["conteudo"],
            "fonte": r["fonte"],
            "score": r["@search.score"],
            "reranker": r["@search.reranker_score"],
        }
        for r in resultados
    ]


pergunta = PERGUNTAS["multissalto"]["pergunta"]
elos = PERGUNTAS["multissalto"]["trechos_esperados"]  # quatro grupos: um trecho de cada documento da cadeia

top5 = buscar_hibrido_com_reranking(pergunta)
ids_top5 = [t["id"] for t in top5]

display(Markdown(f"**Pergunta:** {pergunta}"))
display(pd.DataFrame([
    {
        "posição": i,
        "score RRF": round(t["score"], 4),
        "reranker (0 a 4)": round(t["reranker"], 2),
        "trecho": f"{t['titulo']} — {t['secao']}",
        "é elo da cadeia?": "sim" if any(t["id"] in grupo for grupo in elos) else "",
    }
    for i, t in enumerate(top5, start=1)
]))

fora = ", ".join(f"`{grupo[0]}`" for grupo in elos if not any(i in ids_top5 for i in grupo)) or "nenhum"
display(Markdown(f"**Elos da cadeia no top 5:** {cobertura(ids_top5, elos)}  \n**Ficaram de fora:** {fora}"))
display(Markdown(responder(pergunta, top5)))

### O que observar

- Leia **Elos da cadeia no top 5** antes da resposta. Com 3 de 4, por exemplo, o modelo sabe que a Alfa está suspensa e fornece o 48213-B, e que o ATLAS depende da P-400, mas nenhum trecho diz que a P-400 usa o 48213-B.
- Sem esse elo, a resposta certa é a abstenção. Um modelo que "deduz" a ligação sem trecho que a sustente está chutando, mesmo quando acerta.

### Leitura arquitetural

- Não é falha de configuração: com `k = 50` e um acervo de 50 trechos, todos chegam ao semantic ranker, inclusive o plano da P-400. O ranker lê cada trecho junto com a pergunta, e o trecho da P-400, sozinho, não fala de fornecedor, de suspensão nem de projeto: tende a ficar com nota baixa. Nenhum ajuste de `k`, de `top` ou de pesos garante um trecho que não compartilha nem palavra nem assunto com a pergunta.
- Subconsultas planejadas de antemão (o agêntico do notebook 04) também tendem a tropeçar: a pergunta "quem usa o SKU 48213-B?" só aparece depois de ler o primeiro trecho.
- É o sintoma do degrau 4 do slide 17: **falha em pergunta que exige ligar fatos de documentos diferentes**.

---

# Execução 2 — Indexação do grafo (a parte cara)

O grafo precisa existir antes da primeira pergunta. Na indexação, **cada documento passa pelo modelo de chat**, que devolve as relações que encontrou como triplas (origem, relação, destino), cada uma com o `id` do trecho de onde saiu.

- **Uma chamada por documento:** 14 chamadas ao modelo de chat para este acervo. O índice vetorial do notebook 00 precisou de 4 chamadas de embedding para os mesmos 50 trechos.
- **Structured outputs:** `responses.parse` com um modelo Pydantic (`Extracao`). O serviço garante o formato (um JSON que segue o esquema), não o conteúdo: por isso o código descarta as relações que citam um trecho de fora do documento ou que chegam com um nome vazio.
- **Resolução de entidades:** a mesma entidade aparece escrita de jeitos diferentes em documentos diferentes ("Rolamentos Alfa Ltda." e "Rolamentos Alfa"). A função `normalizar` é a versão ingênua: minúsculas, sem complementos entre parênteses e sem "Ltda." ou "S.A." no fim. É ela que decide quais nomes viram o mesmo nó.
- **Cache:** o resultado vai para `grafo.json`, nesta pasta. Se o arquivo existir, ele é lido e o modelo não é chamado: apague `grafo.json` para reextrair.

In [ ]:
import time

import networkx as nx
from pydantic import BaseModel


class Relacao(BaseModel):
    origem: str
    relacao: str
    destino: str
    trecho_id: str


class Extracao(BaseModel):
    relacoes: list[Relacao]


INSTRUCOES_EXTRACAO = """
Você monta um grafo de conhecimento a partir de documentos internos de uma empresa.
Extraia as relações factuais entre entidades nomeadas e específicas: fornecedores, itens (SKUs), máquinas, projetos, pessoas e áreas.
- Escreva cada entidade como ela aparece no texto, sem complementos entre parênteses (por exemplo: "SKU 12345-X", "Prensa X-100", "Projeto XYZ").
- Não crie entidades genéricas, como "a empresa", "o projeto", "as áreas" ou "o colaborador".
- Use um verbo curto em português para a relação (por exemplo: fornece, usa, depende de, gerencia, suspende).
- Em trecho_id, copie o id da seção de onde a relação foi tirada: o id que aparece entre colchetes no início da seção, sem os colchetes.
- Extraia só o que o texto afirma.
""".strip()


def extrair_relacoes(trechos_do_documento: list[dict]):
    """Uma chamada ao modelo por documento; a resposta segue o esquema Extracao (structured outputs)."""
    secoes = "\n\n".join(f"[{t['id']}] {t['secao']}\n{t['conteudo']}" for t in trechos_do_documento)
    return client.responses.parse(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES_EXTRACAO,
        input=f"DOCUMENTO: {trechos_do_documento[0]['titulo']}\n\n{secoes}",
        text_format=Extracao,
        store=False,
    )


def normalizar(nome: str) -> str:
    """Resolução de entidades ingênua: minúsculas, sem parênteses e sem "Ltda." ou "S.A." no fim."""
    nome = " ".join(re.sub(r"\(.*?\)", " ", nome.casefold()).split())
    return re.sub(r" (ltda|s\.a)\.?$", "", nome)

In [ ]:
CAMINHO_GRAFO = Path.cwd() / "grafo.json"  # o cache da indexação, ao lado do notebook

if CAMINHO_GRAFO.exists():
    extracoes = json.loads(CAMINHO_GRAFO.read_text(encoding="utf-8"))
    print(f"{CAMINHO_GRAFO.name} encontrado: nenhuma chamada ao modelo agora (apague grafo.json para reextrair).")
else:
    extracoes = []
    for doc_id in sorted({t["doc_id"] for t in trechos}):
        inicio = time.perf_counter()
        resposta = extrair_relacoes([t for t in trechos if t["doc_id"] == doc_id])
        relacoes = resposta.output_parsed.relacoes if resposta.output_parsed else []
        extracoes.append({
            "doc_id": doc_id,
            "segundos": round(time.perf_counter() - inicio, 1),
            "tokens_entrada": resposta.usage.input_tokens,
            "tokens_saida": resposta.usage.output_tokens,
            "relacoes": [r.model_dump() for r in relacoes],
        })
        print(f"{doc_id}: relações extraídas = {len(relacoes)}")
    CAMINHO_GRAFO.write_text(json.dumps(extracoes, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Grafo salvo em {CAMINHO_GRAFO}")

grafo = nx.MultiDiGraph()
grafias = {}  # nó normalizado -> como o nome foi escrito nos documentos
descartadas = []  # relações que citam um trecho de outro documento ou que vieram com um nome vazio
for e in extracoes:
    ids_do_documento = {t["id"] for t in trechos if t["doc_id"] == e["doc_id"]}
    for r in e["relacoes"]:
        trecho_id = r["trecho_id"].strip("[] ")  # aceita o id com ou sem os colchetes
        origem, destino = normalizar(r["origem"]), normalizar(r["destino"])
        if trecho_id not in ids_do_documento or not origem or not destino:
            descartadas.append(r)
            continue
        grafo.add_edge(origem, destino, relacao=r["relacao"], trecho_id=trecho_id)
        grafias.setdefault(origem, set()).add(r["origem"])
        grafias.setdefault(destino, set()).add(r["destino"])

display(pd.DataFrame([
    {
        "documento": e["doc_id"],
        "relações extraídas": len(e["relacoes"]),
        "tokens de entrada": e["tokens_entrada"],
        "tokens de saída": e["tokens_saida"],
        "segundos": e["segundos"],
    }
    for e in extracoes
]))
total_entrada = sum(e["tokens_entrada"] for e in extracoes)
total_saida = sum(e["tokens_saida"] for e in extracoes)
print(f"Indexação do grafo: {len(extracoes)} chamadas ao modelo de chat, {total_entrada} tokens de entrada e {total_saida} de saída.")
print(f"Grafo: {grafo.number_of_nodes()} entidades e {grafo.number_of_edges()} relações.")
if descartadas:
    print(f"Relações descartadas (trecho de outro documento ou nome vazio): {len(descartadas)}. Exemplos (até três):")
    for r in descartadas[:3]:
        print(f"   {r['origem']} -{r['relacao']}-> {r['destino']} [{r['trecho_id']}]")

# Primeiro as entidades que juntaram grafias diferentes (a normalização agindo), depois as mais conectadas.
ordem = sorted(grafo, key=lambda n: (len(grafias[n]) > 1, grafo.degree(n)), reverse=True)
display(Markdown("**Entidades do grafo: primeiro as que juntaram grafias diferentes, depois as mais conectadas**"))
display(pd.DataFrame([
    {"entidade (nó)": n, "relações": grafo.degree(n), "como foi escrita": " | ".join(sorted(grafias[n]))}
    for n in ordem[:12]
]))

### O que observar

- **Custo:** no notebook 00, o acervo inteiro virou vetor em 4 chamadas de embedding, só com tokens de entrada. Aqui, cada documento custa uma chamada de chat, e os tokens de saída incluem o raciocínio do modelo: na família gpt-5, raciocínio é cobrado como saída. E a conta se repete: documento alterado é documento reextraído.
- **Resolução de entidades:** nas primeiras linhas da segunda tabela, a coluna **como foi escrita** mostra os nomes que a normalização juntou num nó só (por exemplo, "Rolamentos Alfa" e "Rolamentos Alfa Ltda."). Quando dois nomes da mesma coisa não se juntam (por exemplo, "P-400" e "Prensa P-400"), o grafo se parte em dois, sem erro nenhum, e o caminho da Execução 3 deixa de existir.
- **Entidades mais conectadas:** nomes que aparecem em muitos documentos (uma área, um comitê, uma planta) podem virar atalhos entre assuntos que não têm nada a ver um com o outro.

---

# Execução 3 — Consulta no grafo

Na consulta, o grafo não compara vetores: ele **anda pelas relações**.

1. **Entidades de partida:** os nós do grafo cujo nome aparece na pergunta, depois do mesmo `normalizar` da indexação. É a regra mais simples possível, e é transparente: dá para ver na tela por que cada nó foi escolhido. (O GraphRAG escolhe as entidades por similaridade semântica com a pergunta; aqui é pelo nome, para ficar visível.)
2. **Destinos:** os nós de projeto (nome começando por "projeto").
3. **Caminho:** o mais curto de cada entidade de partida até cada projeto (`nx.shortest_path`), ignorando o sentido das setas: "a Alfa fornece o SKU" e "a prensa usa o SKU" ligam a Alfa à prensa.
4. **Nós genéricos e ordem:** as palavras de `PALAVRAS_GENERICAS` (como "risco" e "projeto"), que sozinhas não identificam nada, não viram partida; e a tabela começa pelos caminhos da entidade de partida mais específica (o nome mais longo), do mais curto ao mais longo: o primeiro segue para a Execução 4.

### O que esperar e por que pode variar

- O caminho até o ATLAS tende a ter 3 saltos: Rolamentos Alfa → SKU 48213-B → Prensa P-400 → Projeto ATLAS.
- Os outros projetos podem aparecer com caminhos mais longos, passando por um nó genérico (um comitê, uma planta, uma área). Existir caminho não é o mesmo que existir risco: por isso a Execução 4 usa só o caminho mais curto.
- Em caso de empate (dois caminhos com o mesmo número de saltos), só um deles segue para a Execução 4, e ele pode passar por um nó genérico em vez do SKU e da prensa: confira a coluna **caminho** antes de ler a resposta.
- Os nomes dos nós, e até a existência do caminho, dependem da extração da Execução 2, que é feita por um modelo e pode mudar se o `grafo.json` for refeito.

In [ ]:
PALAVRAS_GENERICAS = {"empresa", "fornecedor", "projeto", "risco", "suspensão"}  # sozinhas, não identificam nada

pergunta_normalizada = normalizar(pergunta)
na_pergunta = [n for n in grafo if re.search(rf"(?<!\w){re.escape(n)}(?!\w)", pergunta_normalizada)]
partida = [n for n in na_pergunta if n not in PALAVRAS_GENERICAS]
projetos = [n for n in grafo if n.startswith("projeto ")]
nao_direcionado = grafo.to_undirected(as_view=True)  # para achar o caminho, a relação vale nos dois sentidos


def relacoes_entre(a: str, b: str) -> list[dict]:
    """As relações entre dois nós vizinhos, nos dois sentidos, cada uma com o trecho de onde saiu."""
    return [
        {"origem": u, "relacao": d["relacao"], "destino": v, "trecho_id": d["trecho_id"]}
        for u, v in [(a, b), (b, a)]
        for d in (grafo.get_edge_data(u, v) or {}).values()
    ]


def cadeia(caminho: list[str]) -> str:
    """O caminho como texto, com o sentido de cada relação: A -fornece-> B <-usa- C."""
    texto = caminho[0]
    for a, b in zip(caminho, caminho[1:]):
        r = relacoes_entre(a, b)[0]
        texto += f" -{r['relacao']}-> {b}" if r["origem"] == a else f" <-{r['relacao']}- {b}"
    return texto


caminhos = []
for origem in partida:
    for projeto in projetos:
        if nx.has_path(nao_direcionado, origem, projeto):
            caminhos.append(nx.shortest_path(nao_direcionado, origem, projeto))
caminhos.sort(key=lambda c: (-len(c[0]), len(c)))  # primeiro a partida mais específica (nome mais longo), depois menos saltos
caminho = caminhos[0] if caminhos else []  # é ele que vai para o contexto na Execução 4

nomes_partida = ", ".join(partida) or "nenhuma"
genericas = ", ".join(n for n in na_pergunta if n not in partida)
if genericas:
    nomes_partida += f" (palavras genéricas ignoradas: {genericas})"
display(Markdown(f"**Entidades da pergunta encontradas no grafo:** {nomes_partida}"))
if caminhos:
    display(pd.DataFrame([{"projeto": c[-1], "saltos": len(c) - 1, "caminho": cadeia(c)} for c in caminhos]))
else:
    if not partida:
        motivo = "nenhum nó do grafo, fora as palavras genéricas, tem um nome que aparece na pergunta"
    elif not projetos:
        motivo = "nenhum nó do grafo começa por \"projeto\" (o modelo pode ter escrito só \"ATLAS\")"
    else:
        motivo = (
            "não há caminho entre a entidade de partida e os projetos; em geral, a mesma entidade "
            "ficou com dois nomes (por exemplo, \"P-400\" e \"Prensa P-400\")"
        )
    display(Markdown(
        f"**O grafo não ligou a pergunta a nenhum projeto.** Motivo: {motivo}. O problema está na extração da "
        "Execução 2, não na consulta: apague `grafo.json` e rode a Execução 2 de novo para reextrair."
    ))

In [ ]:
import matplotlib.pyplot as plt

vizinhos_da_partida = {v for origem in partida for v in nao_direcionado[origem]}
desenho = nx.DiGraph(grafo.subgraph(set(caminho) | set(partida) | vizinhos_da_partida))  # uma seta por par basta

# O caminho numa linha, da partida até o projeto; as outras relações da partida, numa linha acima.
posicoes = {n: (3 * i, 0) for i, n in enumerate(caminho or partida)}
outros = [n for n in desenho if n not in posicoes]
posicoes |= {n: (1.5 * j - 0.75 * (len(outros) - 1), 1) for j, n in enumerate(outros)}

plt.figure(figsize=(13, 5))
nx.draw_networkx(
    desenho,
    posicoes,
    labels={n: n.replace(" ", "\n") for n in desenho},
    node_color=["#86b6ef" if n in caminho else "#e3e2de" for n in desenho],
    node_size=2400,
    font_size=8,
    edge_color="#8a8984",
)
nx.draw_networkx_edge_labels(desenho, posicoes, edge_labels=nx.get_edge_attributes(desenho, "relacao"), font_size=8)
plt.title(
    "Embaixo, em azul, o caminho mais curto até um projeto; em cima, as outras relações da entidade de partida"
    if caminho
    else "Nenhum caminho até um projeto: só as entidades da pergunta e as relações delas"
)
plt.axis("off")
plt.show()

### O que observar

- Cada salto do caminho é uma relação extraída de um documento diferente: nenhum trecho sozinho contém a cadeia, mas o grafo a monta salto a salto.
- No desenho, a linha de baixo é o caminho, da entidade de partida até o projeto. A seta mostra o sentido da relação como o modelo a extraiu; o caminho foi procurado ignorando esse sentido.
- Se não apareceu caminho, a mensagem acima diz o motivo, que quase sempre está na extração da Execução 2, não na busca: o mesmo item com dois nomes ("P-400" e "Prensa P-400") ou o projeto escrito sem a palavra "Projeto" (só "ATLAS"), que a regra dos destinos não reconhece.

---

# Execução 4 — Resposta com o grafo

Agora o contexto é montado pelo grafo, não pela similaridade:

- as relações do caminho mais curto e as relações da própria entidade de partida (é nelas que está a suspensão);
- os trechos de onde cada relação saiu, buscados pelo `trecho_id`.

As relações entram como um trecho a mais, de id `grafo`, e o `responder` é o mesmo da Execução 1. É o desenho da *local search* do GraphRAG: partir das entidades da pergunta, juntar as relações e as entidades vizinhas e trazer os trechos de origem.

### O que esperar e por que pode variar

- A resposta tende a ligar Rolamentos Alfa → SKU 48213-B → Prensa P-400 → Projeto ATLAS, citando os quatro trechos.
- Se a Execução 3 não achou caminho, o contexto fica só com as relações da Rolamentos Alfa, e o modelo deve se abster.

In [ ]:
pares = list(zip(caminho, caminho[1:])) + [(origem, v) for origem in partida for v in nao_direcionado[origem]]
relacoes_contexto = []
for a, b in pares:
    for r in relacoes_entre(a, b):
        if r not in relacoes_contexto:
            relacoes_contexto.append(r)

por_id = {t["id"]: t for t in trechos}
ids_fonte = list(dict.fromkeys(r["trecho_id"] for r in relacoes_contexto))
trecho_do_grafo = {
    "id": "grafo",
    "titulo": "Grafo de entidades",
    "secao": "relações usadas para responder",
    "conteudo": "\n".join(f"{r['origem']} -{r['relacao']}-> {r['destino']} [{r['trecho_id']}]" for r in relacoes_contexto),
}

display(pd.DataFrame(relacoes_contexto).rename(columns={"relacao": "relação", "trecho_id": "trecho de origem"}))
enviados = ", ".join(f"`{i}`" for i in ids_fonte) or "nenhum"
display(Markdown(
    f"**Trechos de origem enviados ao modelo:** {enviados}  \n"
    f"**Elos da cadeia no contexto:** {cobertura(ids_fonte, elos)} (na Execução 1: {cobertura(ids_top5, elos)})"
))
display(Markdown(responder(pergunta, [trecho_do_grafo] + [por_id[i] for i in ids_fonte])))

### O que observar

- Compare **Elos da cadeia no contexto** com a Execução 1: mesmo modelo, mesmo `responder`, mesma pergunta. Mudou só a recuperação.
- Confira se cada salto da resposta cita o trecho certo. O grafo diz *que* a ligação existe; o trecho de origem é a prova.

### Leitura arquitetural

- **Grafo é indexação cara:** uma chamada de LLM por documento, refeita a cada documento alterado, e uma resolução de entidades frágil, que quebra a cadeia sem dar erro.
- **Em produção:** o Microsoft GraphRAG (Microsoft Research, código aberto) faz a extração em escala, agrupa as entidades em comunidades e resume cada uma. Ele responde com *local search* (a partir das entidades da pergunta, como aqui) e *global search* (perguntas sobre o acervo inteiro, a partir dos resumos das comunidades). Roda **fora do AI Search** (slide 15), e a própria documentação do projeto avisa que a indexação pode sair cara.
- **Só suba para cá por sintoma medido:** uma pergunta de resposta conhecida que o híbrido com reranking não responde, e o gabarito mostrando qual elo faltou. Sem esse número, o grafo é moda.

### Fala sugerida

> “Comece em híbrido com reranking. Suba para agêntico ou grafo por sintoma medido, nunca por moda.”

---

# Quadro final — o que cada execução provou?

| Execução | O que vimos | Decisão arquitetural |
|---|---|---|
| 1 | O híbrido com reranking trouxe parte da cadeia; o elo da P-400 tende a ficar de fora | Medir com pergunta de resposta conhecida antes de subir de degrau |
| 2 | Uma chamada de chat por documento, com tokens de raciocínio, e nomes juntados por uma regra ingênua | Grafo é indexação cara: orçar, guardar em cache e vigiar a resolução de entidades |
| 3 | Da entidade da pergunta até o projeto, salto a salto, cada relação com a sua fonte | A consulta anda por relações, não por similaridade |
| 4 | Mesmo `responder`, outro contexto: a cadeia completa, com citações | Grafo só quando o sintoma medido é ligar fatos de documentos diferentes |

## Frase de fechamento

> **Nenhuma dessas quatro decisões começa por escolher a tecnologia. Todas começam por medir o que está falhando.**

---

# Referências oficiais

- Azure AI Search — consulta híbrida com semantic ranker (k = 50):  
  https://learn.microsoft.com/azure/search/hybrid-search-how-to-query

- Azure AI Search — como o semantic ranker reordena (top 50, nota de 0 a 4):  
  https://learn.microsoft.com/azure/search/semantic-search-overview

- Azure OpenAI — structured outputs (responses.parse com Pydantic):  
  https://learn.microsoft.com/azure/foundry/openai/how-to/structured-outputs

- Microsoft GraphRAG — Getting Started da documentação do projeto (aviso de custo no início da página; indexação, local search e global search):  
  https://microsoft.github.io/graphrag/get_started/